In [ ]:
"""
Baseball Live Totals Pricing Model v2
======================================
Prices live over/under bets using an empirically calibrated
RE24-grounded Poisson run expectancy framework.

Key improvements over v1:
  - Full RE24 matrix (2016-2024 empirical data) replaces flat 0.47/HI
  - base_state parameter for mid-inning runner configurations
  - bet_type: "under" or "over" with correct probability direction
  - runs_needed correctly computed as floor(line) - total for under
  - late_game_discount for competitive innings 7+ (quality relievers)
  - modern_era_contact_discount (K rate 22.7% suppresses multi-run innings)
  - Extras runner-on-2nd corrected: 0.95 -> 1.13

Colab usage:
    exec(open('blowout_over_model.py').read())
    quick_price(4, 3, 8, 1, 10.5, -840, base_state='100', bet_type='under')
    main("table")
    main("demo")
    main("backtest")

Terminal usage:
    python blowout_over_model.py            # interactive
    python blowout_over_model.py --demo
    python blowout_over_model.py --table
    python blowout_over_model.py --backtest
"""

import math
import csv
import os
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional


# ─────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────

# Corrected from X to X (empirical REX bases-empty/X-out = X)
LEAGUE_AVG_RUNS_PER_HALF_INNING = 0.5

DEFAULT_VIG_BUFFER = 0.95  # 個人智慧財產,參數不提供 (placeholder)
TRADE_LOG_PATH     = "blowout_trade_log.csv"

# Late-game competitive discount: innings X-X with quality relievers produce
# ~X-X% fewer runs than context-neutral REX implies.
# Applied only when run_diff <= X. Blowouts (run_diff >= X): no discount.
LATE_GAME_QUALITY_DISCOUNT = 0.90  # 個人智慧財產,參數不提供 (placeholder)

# Modern era (X-X): K rate X% vs X% in early Xs.
# Suppresses multi-run string-together innings.
MODERN_ERA_CONTACT_DISCOUNT = 1.0  # 個人智慧財產,參數不提供 (placeholder)

TRADE_LOG_FIELDS = [
    "date", "game", "inning", "outs", "score_leading", "score_trailing",
    "run_differential", "total_runs_scored", "line", "bet_type", "runs_needed",
    "market_price", "base_state", "relievers_used", "pen_innings_used",
    "back_to_back_arms", "is_dh_game2", "game1_was_blowout",
    "position_player_confirmed", "position_player_probable",
    "trailing_team_wrc_plus", "pds", "tier", "base_lambda", "adj_lambda",
    "true_prob", "fair_price", "market_implied_prob", "edge_pct",
    "signal", "entry_taken", "outcome", "actual_remaining_runs",
    "lambda_error", "notes",
]


# ─────────────────────────────────────────────
# EMPIRICAL REX MATRIX  (X-X, innings X-X)
# Source: FanGraphs / Retrosheet play-by-play
# Excludes Xth/extras (home team walk-off truncation bias)
#
# Key: (base_state, outs)
#   base_state: 'X'=empty 'X'=Xst 'X'=Xnd 'X'=Xrd
#               'X'=Xst&Xnd 'X'=Xst&Xrd 'X'=Xnd&Xrd 'X'=loaded
# Value: expected runs to end of current half-inning
# ─────────────────────────────────────────────

RE24 = {
    ("000", 0): 0.498,  ("000", 1): 0.268,  ("000", 2): 0.106,
    ("100", 0): 0.858,  ("100", 1): 0.512,  ("100", 2): 0.220,
    ("010", 0): 1.130,  ("010", 1): 0.673,  ("010", 2): 0.312,
    ("110", 0): 1.440,  ("110", 1): 0.921,  ("110", 2): 0.414,
    ("001", 0): 1.350,  ("001", 1): 0.937,  ("001", 2): 0.372,
    ("101", 0): 1.720,  ("101", 1): 1.200,  ("101", 2): 0.478,
    ("011", 0): 1.930,  ("011", 1): 1.360,  ("011", 2): 0.548,
    ("111", 0): 2.110,  ("111", 1): 1.540,  ("111", 2): 0.695,
    # Extra innings: automatic runner on Xnd, X outs
    # Corrected from X to X per REX ("X", X)
    ("extras_010", 0): 1.130,
}


# ─────────────────────────────────────────────
# DATA CLASSES
# ─────────────────────────────────────────────

@dataclass
class GameState:
    # Required
    score_leading:  int
    score_trailing: int
    inning:         int      # current inning number
    outs:           int      # X, X, or X
    line:           float    # posted total e.g. X
    market_price:   int      # American odds e.g. -X

    # Bet direction
    bet_type: str = "under"  # "under" or "over"

    # Base state: current runners
    base_state: str = "000"

    # Pitching depletion inputs
    relievers_used:            int   = 0
    pen_innings_used:          float = 0.0
    back_to_back_arms:         int   = 0
    is_dh_game2:               bool  = False
    game1_was_blowout:         bool  = False
    position_player_confirmed: bool  = False
    position_player_probable:  bool  = False

    # Offensive quality
    trailing_team_wrc_plus: int = 100

    # Metadata
    date:  str = field(default_factory=lambda: datetime.today().strftime("%Y-%m-%d"))
    game:  str = ""
    notes: str = ""

    # Computed post-init
    run_differential:  int = field(init=False)
    total_runs_scored: int = field(init=False)
    runs_needed:       int = field(init=False)

    def __post_init__(self):
        self.run_differential  = self.score_leading - self.score_trailing
        self.total_runs_scored = self.score_leading + self.score_trailing

        if self.bet_type == "under":
            # Under cashes if final total <= floor(line)
            # e.g. UX, total=X: budget = floor(X) - X = X
            self.runs_needed = int(math.floor(self.line) - self.total_runs_scored)
        else:
            # Over cashes if final total > line
            # e.g. OX, total=X: need ceil(X - X) = X more runs
            self.runs_needed = math.ceil(self.line - self.total_runs_scored)


@dataclass
class PricingResult:
    pds:                 float
    tier:                str
    base_lambda:         float
    adj_lambda:          float
    true_prob:           float
    fair_price:          int
    market_implied_prob: float
    edge_pct:            float
    signal:              bool
    entry_threshold:     int
    breakdown:           dict


# ─────────────────────────────────────────────
# PITCHING DEPLETION SCORE
# ─────────────────────────────────────────────

def compute_pds(
    relievers_used:            int,
    pen_innings_used:          float,
    back_to_back_arms:         int,
    is_dh_game2:               bool,
    game1_was_blowout:         bool,
    position_player_confirmed: bool,
    position_player_probable:  bool,
) -> float:
    """Pitching Depletion Score: 0-10 scale."""
    score  = relievers_used    * 1.0
    score += pen_innings_used  * 0.5
    score += back_to_back_arms * 1.5
    if is_dh_game2:               score += 2.0
    if game1_was_blowout:         score += 2.0
    if position_player_confirmed: score += 3.0
    elif position_player_probable: score += 1.5
    return min(score, 10.0)


def classify_tier(pds: float) -> str:
    if pds >= 7.0: return "Tier 1 -- Max Depletion"
    if pds >= 4.0: return "Tier 2 -- Meaningful Depletion"
    return "Tier 3 -- Standard Blowout"


# ─────────────────────────────────────────────
# MULTIPLIERS
# ─────────────────────────────────────────────

def depletion_multiplier(pds: float) -> float:
    if pds <= 3: return 1.10
    if pds <= 6: return 1.25
    if pds <= 8: return 1.40
    return 1.60


def differential_suppression(run_diff: int) -> float:
    """Trailing team mental disengagement at large deficits."""
    if run_diff >= 10: return 0.85
    if run_diff >= 7:  return 0.95
    return 1.00


def trailing_team_multiplier(wrc_plus: int) -> float:
    if wrc_plus >= 105: return 1.10
    if wrc_plus >= 90:  return 1.00
    return 0.90


def late_game_discount(inning: int, run_diff: int) -> float:
    """
    Quality-of-pitching discount for competitive late-game situations.
    Innings 7-9, run_diff <= 3: x0.83 (quality relievers suppress scoring)
    Blowouts run_diff >= 7: x1.00 (garbage arms, no discount)
    Mid-range 4-6: x0.90
    Pre-7th: x1.00
    """
    if inning < 7:    return 1.00
    if run_diff >= 7: return 1.00
    if run_diff <= 3: return LATE_GAME_QUALITY_DISCOUNT
    return 0.90


# ─────────────────────────────────────────────
# REX-AWARE LAMBDA CALCULATOR
# ─────────────────────────────────────────────

def compute_base_lambda(
    inning:        int,
    outs:          int,
    base_state:    str = "000",
    total_innings: int = 9,
) -> float:
    """
    Expected remaining runs using empirical RE24 matrix.
    Current half-inning: RE24 lookup for actual base/out state.
    Subsequent half-innings: LEAGUE_AVG_RUNS_PER_HALF_INNING (0.50).
    """
    current_hi_runs = RE24.get((base_state, outs), LEAGUE_AVG_RUNS_PER_HALF_INNING)
    completed_half  = (inning - 1) * 2
    total_half      = total_innings * 2
    full_remaining  = max(total_half - completed_half - 1, 0)
    return current_hi_runs + full_remaining * LEAGUE_AVG_RUNS_PER_HALF_INNING


def remaining_half_innings(inning: int, outs: int, total_innings: int = 9) -> float:
    """Fractional remaining half-innings (for display only)."""
    completed = (inning - 1) * 2
    consumed  = outs / 3.0
    return max(total_innings * 2 - completed - consumed, 0.0)


# ─────────────────────────────────────────────
# POISSON PROBABILITY
# ─────────────────────────────────────────────

def poisson_pmf(k: int, lam: float) -> float:
    if lam <= 0: return 1.0 if k == 0 else 0.0
    return math.exp(-lam) * (lam ** k) / math.factorial(k)


def poisson_cdf(n: int, lam: float) -> float:
    """P(X <= n)"""
    return sum(poisson_pmf(k, lam) for k in range(n + 1))


def prob_at_least_n(n: int, lam: float) -> float:
    """P(X >= n)"""
    if n <= 0: return 1.0
    return 1.0 - poisson_cdf(n - 1, lam)


# ─────────────────────────────────────────────
# AMERICAN ODDS
# ─────────────────────────────────────────────

def prob_to_american(prob: float) -> int:
    if prob >= 1.0: return -100000
    if prob <= 0.0: return  100000
    if prob >= 0.5:
        odds = -(prob / (1 - prob)) * 100
    else:
        odds = ((1 - prob) / prob) * 100
    return int(round(odds / 5) * 5)


def american_to_prob(american: int) -> float:
    if american < 0:
        return abs(american) / (abs(american) + 100)
    return 100 / (american + 100)


def ev_per_unit(true_prob: float, american: int) -> float:
    win_amount = 100 / abs(american) if american < 0 else american / 100
    return true_prob * win_amount - (1 - true_prob) * 1.0


# ─────────────────────────────────────────────
# HARD FILTERS  (blowout over only)
# ─────────────────────────────────────────────

def check_hard_filters(gs: GameState):
    failures = []
    if gs.bet_type == "over":
        if gs.run_differential < 7:
            failures.append(f"Run diff {gs.run_differential} < 7 minimum for blowout over")
        if gs.inning > 7:
            failures.append(f"Inning {gs.inning} > 7 -- too late for depletion to compound")
        if gs.runs_needed < 2:
            failures.append(f"Only {gs.runs_needed} run(s) needed -- near certainty already priced")
    if gs.market_price > 0:
        failures.append("Market price is positive -- check bet direction")
    return len(failures) == 0, failures


# ─────────────────────────────────────────────
# CORE PRICING ENGINE
# ─────────────────────────────────────────────

def price_bet(gs: GameState, vig_buffer: float = DEFAULT_VIG_BUFFER) -> PricingResult:
    """
    Main pricing function. Handles both 'under' and 'over'.

    Under: true_prob = P(X <= runs_needed) where runs_needed = floor(line) - total
    Over:  true_prob = P(X >= runs_needed) where runs_needed = ceil(line - total)
    """
    passes, filter_failures = check_hard_filters(gs)

    pds  = compute_pds(
        gs.relievers_used, gs.pen_innings_used, gs.back_to_back_arms,
        gs.is_dh_game2, gs.game1_was_blowout,
        gs.position_player_confirmed, gs.position_player_probable,
    )
    tier = classify_tier(pds)

    base_lambda = compute_base_lambda(gs.inning, gs.outs, gs.base_state)

    dep_mult    = depletion_multiplier(pds)
    diff_supp   = differential_suppression(gs.run_differential)
    trail_mult  = trailing_team_multiplier(gs.trailing_team_wrc_plus)
    lg_disc     = late_game_discount(gs.inning, gs.run_differential)
    modern_disc = MODERN_ERA_CONTACT_DISCOUNT

    adj_lambda = (base_lambda
                  * dep_mult
                  * diff_supp
                  * trail_mult
                  * lg_disc
                  * modern_disc)

    # Direction-aware probability
    if gs.bet_type == "under":
        true_prob = poisson_cdf(gs.runs_needed, adj_lambda)
    else:
        true_prob = prob_at_least_n(gs.runs_needed, adj_lambda)

    fair_price          = prob_to_american(true_prob)
    entry_threshold     = int(fair_price * vig_buffer)
    market_implied_prob = american_to_prob(gs.market_price)
    edge_pct            = (true_prob - market_implied_prob) * 100
    ev                  = ev_per_unit(true_prob, gs.market_price)

    signal = (passes
              and (gs.market_price >= entry_threshold)
              and (true_prob > market_implied_prob))

    rem_hi = remaining_half_innings(gs.inning, gs.outs)

    breakdown = {
        "filter_failures":          filter_failures,
        "base_state":               gs.base_state,
        "bet_type":                 gs.bet_type,
        "remaining_half_innings":   round(rem_hi, 2),
        "re24_current_hi":          RE24.get((gs.base_state, gs.outs), LEAGUE_AVG_RUNS_PER_HALF_INNING),
        "depletion_multiplier":     dep_mult,
        "differential_suppression": diff_supp,
        "trailing_team_multiplier": trail_mult,
        "late_game_discount":       lg_disc,
        "modern_era_discount":      modern_disc,
        "runs_needed":              gs.runs_needed,
        "ev_per_unit":              round(ev, 4),
        "entry_threshold":          entry_threshold,
    }

    return PricingResult(
        pds=round(pds, 2), tier=tier,
        base_lambda=round(base_lambda, 3),
        adj_lambda=round(adj_lambda, 3),
        true_prob=round(true_prob, 4),
        fair_price=fair_price,
        market_implied_prob=round(market_implied_prob, 4),
        edge_pct=round(edge_pct, 2),
        signal=signal,
        entry_threshold=entry_threshold,
        breakdown=breakdown,
    )


# Legacy alias
price_blowout_over = price_bet


# ─────────────────────────────────────────────
# PROBABILITY LOOKUP TABLE
# ─────────────────────────────────────────────

def print_lookup_table():
    lambdas = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]
    ns      = list(range(1, 8))

    def table(title, prob_fn):
        print("\n" + "=" * 64)
        print(f"  {title}")
        print("=" * 64)
        header = f"  {'lam':>5} | " + " | ".join(f"  N={n}" for n in ns)
        print(header)
        print("  " + "-" * 60)
        for lam in lambdas:
            row = f"  {lam:>5.1f} | " + " | ".join(f"{prob_fn(n, lam)*100:>5.1f}%" for n in ns)
            print(row)
        print("=" * 64)

    table("UNDER  P(X <= N more runs)", poisson_cdf)
    table("OVER   P(X >= N more runs)", prob_at_least_n)
    print()


# ─────────────────────────────────────────────
# RESULT PRINTER
# ─────────────────────────────────────────────

def print_result(gs: GameState, result: PricingResult):
    sep  = "=" * 58
    side = gs.bet_type.upper()

    print(f"\n{sep}")
    print(f"  LIVE TOTALS PRICER v2  --  {side}")
    print(f"  {gs.game or 'Game'} | {gs.date}")
    print(sep)

    print(f"\n  GAME STATE")
    print(f"  Score:          {gs.score_leading}-{gs.score_trailing}  (diff: +{gs.run_differential})")
    print(f"  Inning / Outs:  {gs.inning}th, {gs.outs} out(s)")
    print(f"  Base state:     {gs.base_state}")
    print(f"  Total scored:   {gs.total_runs_scored}")
    print(f"  Line:           {'U' if gs.bet_type == 'under' else 'O'}{gs.line}")
    print(f"  Budget:         <=  {gs.runs_needed} more runs  ({side} cashes)")

    if gs.bet_type == "over":
        print(f"\n  DEPLETION SCORE")
        print(f"  PDS:            {result.pds} / 10  ({result.tier})")
        print(f"  Relievers:      {gs.relievers_used} used, {gs.pen_innings_used} IP")
        print(f"  Back-to-back:   {gs.back_to_back_arms} arms")
        print(f"  DH Game 2:      {'Yes' if gs.is_dh_game2 else 'No'}")
        print(f"  G1 Blowout:     {'Yes' if gs.game1_was_blowout else 'No'}")
        pp = "CONFIRMED" if gs.position_player_confirmed else ("Probable" if gs.position_player_probable else "No")
        print(f"  Pos. player:    {pp}")

    print(f"\n  RUN EXPECTANCY  (RE24-calibrated)")
    print(f"  RE24 curr HI:   {result.breakdown['re24_current_hi']:.3f}  ({gs.base_state}, {gs.outs} out)")
    print(f"  Remaining HI:   {result.breakdown['remaining_half_innings']}")
    print(f"  Base lambda:    {result.base_lambda}")
    if gs.bet_type == "over":
        print(f"  Depl. mult:     x{result.breakdown['depletion_multiplier']}")
        print(f"  Diff. supp:     x{result.breakdown['differential_suppression']}")
        print(f"  Trail. qual:    x{result.breakdown['trailing_team_multiplier']}")
    print(f"  Late-game disc: x{result.breakdown['late_game_discount']}")
    print(f"  Modern disc:    x{result.breakdown['modern_era_discount']}")
    print(f"  Adjusted lambda:{result.adj_lambda}")

    print(f"\n  PRICING")
    print(f"  True prob:      {result.true_prob*100:.2f}%")
    print(f"  Fair price:     {result.fair_price}")
    print(f"  Entry thresh:   {result.entry_threshold}  (fair x {DEFAULT_VIG_BUFFER})")
    print(f"  Market price:   {gs.market_price}")
    print(f"  Mkt implied:    {result.market_implied_prob*100:.2f}%")
    print(f"  Edge:           {result.edge_pct:+.2f}pp")
    print(f"  EV / unit:      {result.breakdown['ev_per_unit']:+.4f}")

    if result.breakdown["filter_failures"]:
        print(f"\n  WARNING: FILTER FAILURES")
        for f in result.breakdown["filter_failures"]:
            print(f"    - {f}")

    verdict = "SIGNAL -- ENTER" if result.signal else "NO SIGNAL -- PASS"
    print(f"\n  {verdict}")
    print(f"{sep}\n")


# ─────────────────────────────────────────────
# QUICK PRICE  (programmatic / Colab)
# ─────────────────────────────────────────────

def quick_price(
    score_leading:             int,
    score_trailing:            int,
    inning:                    int,
    outs:                      int,
    line:                      float,
    market_price:              int,
    bet_type:                  str   = "under",
    base_state:                str   = "000",
    relievers_used:            int   = 0,
    pen_innings_used:          float = 0.0,
    back_to_back_arms:         int   = 0,
    is_dh_game2:               bool  = False,
    game1_was_blowout:         bool  = False,
    position_player_confirmed: bool  = False,
    position_player_probable:  bool  = False,
    trailing_team_wrc_plus:    int   = 100,
    game:                      str   = "",
    notes:                     str   = "",
    verbose:                   bool  = True,
) -> PricingResult:
    """
    Price any live total bet.

    base_state encoding:
        '000' bases empty    '100' runner on 1st
        '010' runner on 2nd  '001' runner on 3rd
        '110' 1st and 2nd    '101' 1st and 3rd
        '011' 2nd and 3rd    '111' bases loaded

    Examples:
        # Competitive under, runner on Xst, X out
        quick_price(4, 3, 8, 1, 10.5, -840, base_state='100')

        # Blowout over
        quick_price(10, 0, 6, 0, 12.5, -685, bet_type='over',
                    relievers_used=3, is_dh_game2=True, game1_was_blowout=True)

        # Extra innings under, automatic runner on Xnd
        quick_price(1, 1, 10, 0, 6.5, -805, base_state='010')
    """
    gs = GameState(
        score_leading=score_leading, score_trailing=score_trailing,
        inning=inning, outs=outs, line=line, market_price=market_price,
        bet_type=bet_type, base_state=base_state,
        relievers_used=relievers_used, pen_innings_used=pen_innings_used,
        back_to_back_arms=back_to_back_arms,
        is_dh_game2=is_dh_game2, game1_was_blowout=game1_was_blowout,
        position_player_confirmed=position_player_confirmed,
        position_player_probable=position_player_probable,
        trailing_team_wrc_plus=trailing_team_wrc_plus,
        game=game, notes=notes,
    )
    result = price_bet(gs)
    if verbose:
        print_result(gs, result)
    return result


# ─────────────────────────────────────────────
# TRADE LOG
# ─────────────────────────────────────────────

def init_trade_log():
    if not os.path.exists(TRADE_LOG_PATH):
        with open(TRADE_LOG_PATH, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=TRADE_LOG_FIELDS).writeheader()
        print(f"  Trade log created: {TRADE_LOG_PATH}")


def log_trade(
    gs: GameState,
    result: PricingResult,
    entry_taken: bool,
    outcome: Optional[str] = None,
    actual_remaining_runs: Optional[int] = None,
):
    init_trade_log()
    lambda_error = (round(actual_remaining_runs - result.adj_lambda, 3)
                    if actual_remaining_runs is not None else None)
    row = {
        "date": gs.date, "game": gs.game,
        "inning": gs.inning, "outs": gs.outs,
        "score_leading": gs.score_leading, "score_trailing": gs.score_trailing,
        "run_differential": gs.run_differential,
        "total_runs_scored": gs.total_runs_scored,
        "line": gs.line, "bet_type": gs.bet_type,
        "runs_needed": gs.runs_needed, "market_price": gs.market_price,
        "base_state": gs.base_state,
        "relievers_used": gs.relievers_used,
        "pen_innings_used": gs.pen_innings_used,
        "back_to_back_arms": gs.back_to_back_arms,
        "is_dh_game2": gs.is_dh_game2,
        "game1_was_blowout": gs.game1_was_blowout,
        "position_player_confirmed": gs.position_player_confirmed,
        "position_player_probable": gs.position_player_probable,
        "trailing_team_wrc_plus": gs.trailing_team_wrc_plus,
        "pds": result.pds, "tier": result.tier,
        "base_lambda": result.base_lambda, "adj_lambda": result.adj_lambda,
        "true_prob": result.true_prob, "fair_price": result.fair_price,
        "market_implied_prob": result.market_implied_prob,
        "edge_pct": result.edge_pct, "signal": result.signal,
        "entry_taken": entry_taken,
        "outcome": outcome or "pending",
        "actual_remaining_runs": actual_remaining_runs,
        "lambda_error": lambda_error,
        "notes": gs.notes,
    }
    with open(TRADE_LOG_PATH, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=TRADE_LOG_FIELDS).writerow(row)
    print(f"  Trade logged -> {TRADE_LOG_PATH}")


# ─────────────────────────────────────────────
# BACKTEST
# ─────────────────────────────────────────────

def run_backtest():
    if not os.path.exists(TRADE_LOG_PATH):
        print("  No trade log found."); return
    with open(TRADE_LOG_PATH) as f:
        trades = [r for r in csv.DictReader(f) if r["outcome"] != "pending"]
    if not trades:
        print("  No completed trades."); return

    def stats(subset, label):
        if not subset:
            print(f"  {label}: no trades"); return
        wins    = sum(1 for t in subset if t["outcome"] == "win")
        n       = len(subset)
        avg_edge = sum(float(t["edge_pct"]) for t in subset) / n
        avg_ev   = sum(float(t["ev_per_unit"]) for t in subset) / n
        errs = [float(t["lambda_error"]) for t in subset if t["lambda_error"]]
        print(f"\n  {label}")
        print(f"    Trades:      {n}")
        print(f"    Win rate:    {wins/n*100:.1f}%  ({wins}/{n})")
        print(f"    Avg edge:    {avg_edge:+.2f}pp")
        print(f"    Avg EV/unit: {avg_ev:+.4f}")
        if errs:
            me = sum(errs) / len(errs)
            print(f"    Avg lam err: {me:+.3f}  (+ = model underestimated)")

    sep = "=" * 56
    print(f"\n{sep}\n  BACKTEST RESULTS\n{sep}")
    stats(trades, "ALL TRADES")
    stats([t for t in trades if t["signal"] == "True"],      "SIGNAL = TRUE")
    stats([t for t in trades if t["entry_taken"] == "True"], "ENTERED ONLY")
    for lbl in ["Tier 1", "Tier 2", "Tier 3"]:
        stats([t for t in trades if t["tier"].startswith(lbl)], lbl.upper())
    for bt in ["under", "over"]:
        stats([t for t in trades if t.get("bet_type") == bt], bt.upper())

    errs = [float(t["lambda_error"]) for t in trades if t["lambda_error"]]
    if errs:
        me = sum(errs) / len(errs)
        print(f"\n  LAMBDA CALIBRATION  mean error: {me:+.3f}")
        if   me >  0.3: print("    -> Raise depletion multipliers ~10%")
        elif me < -0.3: print("    -> Raise differential suppression weights")
        else:           print("    -> Model reasonably calibrated")
    print(f"\n{sep}\n")


# ─────────────────────────────────────────────
# INTERACTIVE MODE
# ─────────────────────────────────────────────

def _get_int(prompt, default=None):
    while True:
        v = input(prompt).strip()
        if v == "" and default is not None: return default
        try: return int(v)
        except ValueError: print("    Enter a whole number.")

def _get_float(prompt, default=None):
    while True:
        v = input(prompt).strip()
        if v == "" and default is not None: return default
        try: return float(v)
        except ValueError: print("    Enter a number.")

def _get_bool(prompt):
    while True:
        v = input(prompt).strip().lower()
        if v in ("y","yes","1","true"):  return True
        if v in ("n","no","0","false"):  return False
        print("    Enter y or n.")

def _get_str(prompt, options=None, default=None):
    while True:
        v = input(prompt).strip().lower()
        if v == "" and default is not None: return default
        if options is None or v in options: return v
        print(f"    Options: {options}")


def interactive_mode():
    print("\n" + "=" * 58)
    print("  LIVE TOTALS PRICER v2 -- Interactive Mode")
    print("=" * 58)

    print("\n  GAME STATE")
    score_leading  = _get_int("  Score (leading team):  ")
    score_trailing = _get_int("  Score (trailing team): ")
    inning         = _get_int("  Current inning: ")
    outs           = _get_int("  Outs (0/1/2): ", default=0)
    line           = _get_float("  Line (e.g. 10.5): ")
    bet_type       = _get_str("  Bet type (under/over): ", ["under","over"], "under")
    base_state     = _get_str("  Base state [000/100/010/110/001/101/011/111]: ", default="000")
    market_price   = _get_int("  Market price (e.g. -840): ")

    relievers_used = 0; pen_innings_used = 0.0; back_to_back_arms = 0
    is_dh_game2 = False; game1_was_blowout = False
    position_player_conf = False; position_player_prob = False

    if bet_type == "over":
        print("\n  DEPLETION INPUTS")
        relievers_used    = _get_int("  Relievers used (leading): ", default=0)
        pen_innings_used  = _get_float("  Pen IP used (leading): ", default=0.0)
        back_to_back_arms = _get_int("  Back-to-back arms: ", default=0)
        is_dh_game2       = _get_bool("  DH Game 2? (y/n): ")
        if is_dh_game2:
            game1_was_blowout = _get_bool("  G1 also blowout? (y/n): ")
        position_player_conf = _get_bool("  Position player confirmed? (y/n): ")
        if not position_player_conf:
            position_player_prob = _get_bool("  Position player probable? (y/n): ")

    trailing_wrc = _get_int("  Trailing team wRC+ (default 100): ", default=100)

    print("\n  METADATA (optional -- press Enter to skip)")
    game  = input("  Game label (e.g. BAL@NYY): ").strip()
    notes = input("  Notes: ").strip()

    gs = GameState(
        score_leading=score_leading, score_trailing=score_trailing,
        inning=inning, outs=outs, line=line, market_price=market_price,
        bet_type=bet_type, base_state=base_state,
        relievers_used=relievers_used, pen_innings_used=pen_innings_used,
        back_to_back_arms=back_to_back_arms,
        is_dh_game2=is_dh_game2, game1_was_blowout=game1_was_blowout,
        position_player_confirmed=position_player_conf,
        position_player_probable=position_player_prob,
        trailing_team_wrc_plus=trailing_wrc, game=game, notes=notes,
    )
    result = price_bet(gs)
    print_result(gs, result)
    print_lookup_table()

    if _get_bool("  Log this trade? (y/n): "):
        entry_taken = _get_bool("  Did you enter? (y/n): ")
        outcome = actual_runs = None
        if _get_bool("  Result available? (y/n): "):
            raw = input("  Outcome (win/loss): ").strip().lower()
            outcome = raw if raw in ("win","loss") else None
            raw2 = input("  Actual remaining runs (or Enter to skip): ").strip()
            actual_runs = int(raw2) if raw2 else None
        log_trade(gs, result, entry_taken, outcome, actual_runs)


# ─────────────────────────────────────────────
# DEMO
# ─────────────────────────────────────────────

def run_demo():
    print("\n--- Demo 1: Blowout OVER -- DH G2, 10-0 mid-6th, O12.5 -685 ---")
    quick_price(10, 0, 6, 0, 12.5, -685, bet_type="over",
                relievers_used=3, pen_innings_used=4.0, back_to_back_arms=1,
                is_dh_game2=True, game1_was_blowout=True,
                trailing_team_wrc_plus=95, game="DH G2 blowout")

    print("\n--- Demo 2: Competitive UNDER -- BAL@NYY 3-4 top8 R1st 1out U10.5 -840 ---")
    quick_price(4, 3, 8, 1, 10.5, -840, bet_type="under",
                base_state="100", game="BAL@NYY")

    print("\n--- Demo 3: Large budget UNDER -- AZ@CHC 2-6 top7 U14.5 -835 ---")
    quick_price(6, 2, 7, 0, 14.5, -835, bet_type="under", game="AZ@CHC")

    print("\n--- Demo 4: Extra innings UNDER -- 1-1 bot10 runner 2nd U6.5 -805 ---")
    quick_price(1, 1, 10, 0, 6.5, -805, bet_type="under",
                base_state="010", game="Extra innings")


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

def main(mode="interactive"):
    """
    Colab:
        main()              interactive prompts
        main("demo")        run demo trades
        main("table")       probability lookup tables
        main("backtest")    analyse trade log
    """
    {"backtest": run_backtest, "demo": run_demo,
     "table": print_lookup_table, "interactive": interactive_mode}.get(
        mode, interactive_mode)()


if __name__ == "__main__":
    import sys
    is_colab = any("jupyter" in a or "kernel" in a for a in sys.argv[1:])
    if not is_colab:
        import argparse
        p = argparse.ArgumentParser(description="Live Totals Pricer v2")
        p.add_argument("--demo",     action="store_true")
        p.add_argument("--table",    action="store_true")
        p.add_argument("--backtest", action="store_true")
        a = p.parse_args()
        if   a.demo:     run_demo()
        elif a.table:    print_lookup_table()
        elif a.backtest: run_backtest()
        else:            interactive_mode()